# 00.6 — Git and GitHub for MATLAB users

Version control is the biggest workflow upgrade this curriculum offers. If your MATLAB projects have ever contained `analysis_final.m`, `analysis_final_v2.m`, and `analysis_final_v2_REALLY_FINAL.m` — this notebook replaces that entire genre of filename with one tool.

You will learn:

* The git mental model: snapshots, the three areas, the history graph.
* The daily loop: `status` → `add` → `commit` → `log` / `diff`.
* `.gitignore` — and the deep-learning-specific rules that keep 10 GB of checkpoints out of your history.
* Branches, merges, and undoing mistakes.
* GitHub: remotes, push/pull, pull requests.
* How this repo uses all of it.

Every git command below **actually runs** in a scratch repo under `/tmp` — re-run cells freely.

**Prerequisite:** [00.5 the command line](00.5_the_command_line_for_matlab_users.ipynb).

## Section 1 — What MATLAB does

MATLAB has no built-in habit of version control. The common workflows are:

* Save-As with version suffixes (`_v2`, `_final`, `_final2`) — no record of *what* changed or *why*.
* A shared network drive where the newest timestamp wins.
* (MATLAB *does* ship git integration in the Current Folder browser — right-click → Source Control — but few labs turn it on.)

The costs show up exactly when stakes are highest: "which version produced the figure in the paper?", "what did I change since the run that worked?", "my collaborator and I both edited the same file."

Git answers all three with one model — and it's the same tool this entire Python project is built on, so you'll use it daily.

## Section 2 — The concepts you need

### 2.1 — The mental model: snapshots + three areas

A git repository is a **history of snapshots** (commits). Each commit records: the complete state of your tracked files, a message, an author, a timestamp, and a pointer to its parent commit(s).

Between "I edited a file" and "it's in history" there's one intermediate stop — the **staging area** — which is what trips everyone up on day one. The figure below is the map to keep in your head:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 4))

areas = [
    (1.8, "Working tree", "your files as they\nare on disk right now", "#ffe4cc"),
    (5.5, "Staging area", "the set of changes\nqueued for the next commit", "#cce4ff"),
    (9.2, "History", "immutable chain\nof committed snapshots", "#e6f0d0"),
]
for x, title, sub, color in areas:
    rect = mpatches.FancyBboxPatch((x - 1.5, 1.0), 3.0, 1.8,
        boxstyle="round,pad=0.08", facecolor=color, edgecolor="black", linewidth=1.4)
    ax.add_patch(rect)
    ax.text(x, 2.35, title, ha="center", fontsize=13, fontweight="bold")
    ax.text(x, 1.6, sub, ha="center", fontsize=10, color="#444")

def arrow(x1, x2, y, label, color):
    ax.annotate("", xy=(x2, y), xytext=(x1, y),
        arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.text((x1 + x2) / 2, y + 0.16, label, ha="center", fontsize=11,
        family="monospace", color=color, fontweight="bold")

arrow(3.35, 3.95, 3.25, "git add", "#3366aa")
arrow(7.05, 7.65, 3.25, "git commit", "#447722")
arrow(7.65, 3.35, 0.55, "git restore  (undo, pulls a file back out)", "#aa3333")

ax.text(5.5, 4.0, "the daily loop", ha="center", fontsize=11, style="italic", color="#666")
ax.set_xlim(0, 11); ax.set_ylim(0.2, 4.4)
ax.axis("off")
plt.tight_layout(); plt.show()
print("edit files → git add (choose what goes in) → git commit (snapshot it, forever)")

Why the staging area exists: it lets you commit *some* of your edits (the bug fix) without committing *others* (the debugging print statements) — you choose per file, or even per hunk, what enters the snapshot.

### 2.2 — The daily loop, live

Let's create a real repository and drive the loop. (We use `git -C <dir>` — "run git as if in that directory" — so the notebook's fresh-subshell rule from 00.5 doesn't bite us.)

In [ ]:
REPO = "/tmp/ndd_git_demo"
!rm -rf {REPO} && mkdir -p {REPO}
!git -C {REPO} init -b main
# Give the demo repo its own identity (normally set once globally:
#   git config --global user.name "You" / user.email "you@lab.edu")
!git -C {REPO} config user.name "Demo User" && git -C {REPO} config user.email "demo@example.com"
print("repo created")

In [ ]:
# Create a file, then ask git what it sees
with open(f"{REPO}/analysis.py", "w") as f:
    f.write("import numpy as np\n\ndef decode(x):\n    return x.mean()\n")
!git -C {REPO} status --short

`??` = untracked (git sees the file but isn't versioning it yet). Stage and commit it:

In [ ]:
!git -C {REPO} add analysis.py
!git -C {REPO} status --short
!git -C {REPO} commit -m "Add decoder skeleton"
!git -C {REPO} log --oneline

The commit message matters: it's the searchable record of *why*. This repo's history is a good example of messages that explain intent — run `git log --oneline -10` in the real repo and compare with `analysis_final_v2.m` as documentation.

Now modify the file and see what git shows you **before** you commit:

In [ ]:
with open(f"{REPO}/analysis.py", "a") as f:
    f.write("\ndef decode_std(x):\n    return x.std()\n")
!git -C {REPO} diff

`git diff` is the answer to "what did I change since the version that worked?" — line-level, always available, no `_v2` files needed. Commit the change:

In [ ]:
!git -C {REPO} add analysis.py && git -C {REPO} commit -m "Add std-based decoder variant"
!git -C {REPO} log --oneline

### 2.3 — `.gitignore`: what does NOT belong in history

Git should track **source, configs, and docs** — things humans write. It should NOT track things machines generate, and for deep-learning projects that rule has teeth, because the generated things are enormous:

```gitignore
# ---- Python machinery ----
__pycache__/
*.pyc
.venv/
.ipynb_checkpoints/

# ---- Deep learning artifacts (the important ones!) ----
data/            # raw datasets — belong on a data server, not in git
results/         # training outputs
*.pt             # model checkpoints
*.ckpt
*.mat            # large binary data files
wandb/           # experiment-tracker caches

# ---- Editor noise ----
.DS_Store
.vscode/
```

One 500 MB checkpoint committed by accident bloats every future clone of the repository **forever** (history keeps it even after you delete the file). This is the single most expensive beginner mistake — the `.gitignore` is your guardrail, so write it FIRST, before the first commit.

In [ ]:
# This repo's real .gitignore — data, results, venv, caches all excluded:
!head -30 ../../.gitignore

### 2.4 — Branches: parallel versions without parallel folders

A **branch** is a movable label pointing at a commit. Creating one is instant and costless — it's the git answer to copying the whole project folder to try something risky.

In [ ]:
!git -C {REPO} switch -c experiment-pca
with open(f"{REPO}/analysis.py", "a") as f:
    f.write("\ndef decode_pca(x):\n    return x  # TODO\n")
!git -C {REPO} add -A && git -C {REPO} commit -m "Start PCA decoder experiment"
!git -C {REPO} log --oneline --all

In [ ]:
# The experiment lives ONLY on its branch — switch back and the file reverts:
!git -C {REPO} switch main
!tail -3 {REPO}/analysis.py

In [ ]:
# Happy with the experiment? Merge it into main:
!git -C {REPO} merge experiment-pca
!tail -3 {REPO}/analysis.py

### 2.5 — Undoing things

The three undo tools, from safest to most destructive:

| Situation | Command | Effect |
|---|---|---|
| Discard uncommitted edits to a file | `git restore file.py` | file returns to last committed state (edits GONE) |
| Un-stage a file (keep the edits) | `git restore --staged file.py` | reverses `git add` |
| Undo a committed change, safely | `git revert <commit>` | makes a NEW commit that reverses it — history preserved |
| Rewind history (dangerous) | `git reset --hard <commit>` | moves the branch back; later commits orphaned. Never on shared branches. |

Rule of thumb: `revert` for anything already pushed/shared, `reset` only for local cleanup you're sure about, and when truly lost — `git reflog` shows everywhere HEAD has been and can rescue almost anything.

### 2.6 — GitHub: sharing the history

Everything so far was local. **GitHub** hosts a copy of your repository (a **remote**, conventionally named `origin`) so collaborators and cluster machines can reach it.

```bash
git clone git@github.com:cgerrity/neural_data_decoding.git   # copy a hosted repo (SSH form)
git push                    # upload your new commits to the remote
git pull                    # download others' commits and merge them in
git fetch                   # download without merging (look first)
```

**Authentication**: two options. HTTPS (`https://github.com/...`) prompts for a token; SSH (`git@github.com:...`) uses a key pair you set up once — [GitHub's SSH key guide](https://docs.github.com/en/authentication/connecting-to-github-with-ssh) takes ~5 minutes and is worth it.

**The collaboration loop** (how changes get reviewed):

1. Branch: `git switch -c fix-nan-handling`
2. Commit your work, `git push -u origin fix-nan-handling`
3. Open a **pull request** (PR) on GitHub — a proposal to merge your branch, with a diff view and comment threads
4. Review, revise (new commits update the PR automatically), merge.

**Issues** are GitHub's built-in tracker for bugs/ideas; a **fork** is your own copy of someone else's repo (the contribution path when you lack write access). For a one-person research project you may skip PRs and push to `main` directly — but the branch-and-PR loop is the norm in shared codebases, and reviewers can only comment on what you actually pushed.

## Section 3 — The `neural_data_decoding` implementation

How this repo uses everything above:

* **History as documentation** — each milestone landed as descriptive commits; `git log --oneline` reads as a project chronology. When HANDOFF.md says "parity verified in commit b2d1c33", `git show b2d1c33` reproduces exactly what changed.
* **`.gitignore`** keeps `results/`, `.venv/`, and caches out (you inspected it above). `results/` is ignored wholesale — even the sample `.mat` fixtures the curriculum uses live only on the development machine, because 8 MB of binary ephys data doesn't belong in history. (When you genuinely need to version an exception to an ignore rule, `git add -f` overrides it — use sparingly.)
* **The nbstripout filter** ([notebooks/README](../README.md)) is a git *content filter*: on commit, notebook outputs are stripped automatically, so diffs show only real content changes. Filters are declared in `.gitattributes` (tracked) and activated per-clone with `nbstripout --install`.
* **Remote on GitHub** — the sync point between the development machine and the ACCRE cluster: push from the laptop, `git pull` on the cluster, submit jobs.

## Section 4 — Hands-on exercises

### Exercise 4.1 — cause and read a merge conflict

The one git experience everyone fears. Cause one deliberately in the scratch repo — then see it's just text markers:

In [ ]:
# Two branches edit the SAME line differently:
!git -C {REPO} switch -c version-a >/dev/null 2>&1
with open(f"{REPO}/analysis.py", "w") as f:
    f.write("THRESHOLD = 0.5\n")
!git -C {REPO} commit -am "Set threshold to 0.5" >/dev/null

!git -C {REPO} switch main >/dev/null 2>&1
with open(f"{REPO}/analysis.py", "w") as f:
    f.write("THRESHOLD = 0.9\n")
!git -C {REPO} commit -am "Set threshold to 0.9" >/dev/null

!git -C {REPO} merge version-a; true

In [ ]:
# The 'conflict' is just both versions written into the file with markers:
print(open(f"{REPO}/analysis.py").read())

To resolve: edit the file to the version you want (delete the `<<<<<<<`/`=======`/`>>>>>>>` marker lines), then `git add` + `git commit`. That's the whole ritual:

In [ ]:
with open(f"{REPO}/analysis.py", "w") as f:
    f.write("THRESHOLD = 0.7  # split the difference\n")
!git -C {REPO} add analysis.py && git -C {REPO} commit -m "Resolve threshold conflict at 0.7"
!git -C {REPO} log --oneline -4

### Exercise 4.2 — explore the real repo's history

In a terminal at the repo root, find: (a) the commit that introduced `MatFileTrialDataset`, (b) what files it touched. Hints: `git log --oneline -- src/neural_data_decoding/data/mat_dataset.py` shows commits touching a path; `git show --stat <sha>` shows a commit's files.

## Section 5 — Common errors

### `fatal: not a git repository`

You're not inside a repo (`pwd`, then `cd` to one) — or you meant to `git init`.

### `Please tell me who you are` on first commit

Git needs an identity for the commit record: `git config --global user.name "You"` and `git config --global user.email "you@lab.edu"` — once per machine.

### `! [rejected] ... failed to push some refs`

The remote has commits you don't have (a collaborator pushed, or you edited on GitHub). `git pull` first, resolve any conflict, push again.

### "You are in 'detached HEAD' state"

You checked out a specific commit rather than a branch — fine for looking around; to keep new work from here, `git switch -c rescue-branch`. To leave: `git switch main`.

### Accidentally committed a huge file

*Before pushing:* `git reset --soft HEAD~1`, unstage the file, add it to `.gitignore`, recommit. *After pushing:* the history itself must be rewritten (`git filter-repo`) — genuinely painful, which is why the `.gitignore`-first habit matters.

### Merge conflict panic

Nothing is lost. The conflicted file contains both versions between markers (you saw this above); `git status` lists every conflicted file and `git merge --abort` cancels the whole merge if you want to regroup.

## Section 6 — Further reading

- [Pro Git book](https://git-scm.com/book/en/v2) — free, canonical; chapters 1–3 cover everything here in depth.
- [Oh Shit, Git!?!](https://ohshitgit.com/) — recovery recipes for every classic mistake, profanity included.
- [GitHub's Hello World guide](https://docs.github.com/en/get-started/start-your-journey/hello-world) — the PR loop, hands-on.
- [Learn Git Branching](https://learngitbranching.js.org/) — interactive visual sandbox for branch intuition.

Next up: **[00.7 pip, packaging, and project anatomy](00.7_pip_packaging_and_project_anatomy.ipynb)**.